In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from ipywidgets import interact, widgets
import xarray as xr

OMBraw = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/cruise/2024_KVS_deployment.nc')
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM.nc')

# Define the time range for the plot
start_time = pd.to_datetime("2024-04-18")
end_time = pd.to_datetime("2024-04-25")

# List of buoy numbers
buoy_list = [30, 7, 17, 32, 27, 18, 26]
#buoy_list = [30, 7, 1, 17, 32, 10, 8, 23, 18, 26, 9, 13]

# Create a colormap (light blue to dark blue)
colors = cm.Spectral(np.linspace(0, 1, len(buoy_list)))

# Interactive function to update the plots
def update_plot(specific_time_str):
    specific_time = pd.to_datetime(specific_time_str)
    
    # Create a figure with three subplots (1 for raw data, 1 for model data, 1 for spectra)
    fig, axes = plt.subplots(3, 1, figsize=(10, 12), gridspec_kw={'height_ratios': [2, 2, 1]})
    axes = axes.ravel()

    # Subplot 1: Time series plot for raw data (pHs0)
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        axes[0].scatter(time_series, OMBraw.pHs0[buoyno, :], label=str(buoyno), color=color)
    # Add a grey stripe at the specific time
    stripe_width = pd.Timedelta(hours=1)
    stripe_start = specific_time - stripe_width / 2
    stripe_end = specific_time + stripe_width / 2
    axes[0].add_patch(Rectangle(
        (stripe_start, 0.01),
        stripe_width,
        10,
        color='grey',
        alpha=0.2,
        zorder=0
    ))

    axes[0].set_xlim([start_time, end_time])
    axes[0].set_ylim([0.01, 10])
    axes[0].set_yscale('log')
    axes[0].set_xlabel("Time", fontsize=14)
    axes[0].set_ylabel("Significant Wave Height [m]", fontsize=14)
    axes[0].legend(title="Buoy")
    axes[0].grid(True)

    # Subplot 2: Time series plot for model data (MODHs0)
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        #axes[1].scatter(time_series, OMBwave.MODHs0[0, buoyno, :], label=f'MF-{buoyno}', color=color, marker='*')
        axes[1].scatter(time_series, OMBwave.MODHs0[1, buoyno, :], label=f'MET-{buoyno}', color=color, marker='.')
    
    axes[1].set_xlim([start_time, end_time])
    axes[1].set_ylim([0.01, 10])
    axes[1].set_yscale('log')
    axes[1].set_xlabel("Time", fontsize=14)
    axes[1].set_ylabel("Significant Wave Height [m]", fontsize=14)
    axes[1].legend(title="Model Buoy")
    axes[1].grid(True)

    # Subplot 3: Spectra plot at a specific time
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        specific_time_np = np.datetime64(specific_time)
        
        # Find the closest time step and ensure it is within 2 hours
        time_diffs = np.abs(time_series - specific_time_np)
        tt = np.argmin(time_diffs)
        
        # Check if the closest time step is within 2 hours of the specific time
        if time_diffs[tt] <= np.timedelta64(2, 'h'):
            spectra = OMBraw.processed_elevation_energy_spectrum[buoyno, tt, :]
            freqs = OMBraw.frequencies_waves_imu.values
            axes[2].plot(freqs, spectra, label=f"Buoy {buoyno}", color=color)
        else:
            # If no valid time step, skip plotting for this buoy
            axes[2].plot([], [], label=f"Buoy {buoyno} (No valid time)", color=color)

    axes[2].set_xlabel("Frequency [Hz]", fontsize=14)
    axes[2].set_ylabel("Spectral Energy", fontsize=14)
    axes[2].set_yscale('log')
    axes[2].grid(True)
    axes[2].legend(title="Buoy", loc=4)

    plt.tight_layout()
    plt.show()

# Create an interactive widget
time_slider = widgets.SelectionSlider(
    options=pd.date_range(start=start_time, end=end_time, freq='H').strftime('%Y-%m-%d %H:%M').tolist(),
    description='Time:',
    continuous_update=False
)

# Display the interactive widget
interact(update_plot, specific_time_str=time_slider)


/home/maltem/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
Widget Javascript not detected.  It may not be installed or enabled properly. Reconnecting the current kernel may help.


<function __main__.update_plot(specific_time_str)>

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from ipywidgets import interact, widgets
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
modelno=0
# Load datasets
OMBraw = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/cruise/2024_KVS_deployment.nc')
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM.nc')

# Define the time range for the plot
start_time = pd.to_datetime("2024-04-18")
end_time = pd.to_datetime("2024-04-25")

# Define buoy lists
buoy_list_1 = [30, 7, 1, 17, 32, 10, 8, 23, 18, 26, 9, 13]
buoy_list_2 = [30, 7, 1, 17, 22, 2, 27, 25, 4]
buoy_list_3 = [30, 7, 17, 32, 27, 18, 26]

# Create a colormap (light blue to dark blue)
colors = cm.Spectral(np.linspace(0, 1, len(buoy_list_1)))  # Default list (buoy_list_1)

# Interactive function to update the plots
def update_plot_with_positions(specific_time_str, selected_buoy_list):
    specific_time = pd.to_datetime(specific_time_str)

    # Select buoy list based on the dropdown selection
    if selected_buoy_list == 'List 1':
        buoy_list = buoy_list_1
    elif selected_buoy_list == 'List 2':
        buoy_list = buoy_list_2
    elif selected_buoy_list == 'List 3':
        buoy_list = buoy_list_3

    # Update colors based on the selected list
    colors = cm.Spectral(np.linspace(0, 1, len(buoy_list)))

    # Create a figure with four subplots (adding one for buoy positions)
    fig, axes = plt.subplots(4, 1, figsize=(10, 16), gridspec_kw={'height_ratios': [2, 2, 1, 2]})
    axes = axes.ravel()

    # Subplot 1: Time series plot for raw data (pHs0)
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        axes[0].scatter(time_series, OMBraw.pHs0[buoyno, :], label=str(buoyno), color=color)
    
    stripe_width = pd.Timedelta(hours=1)
    stripe_start = specific_time - stripe_width / 2
    stripe_end = specific_time + stripe_width / 2
    axes[0].add_patch(Rectangle(
        (stripe_start, 0.01),
        stripe_width,
        10,
        color='grey',
        alpha=0.2,
        zorder=0
    ))

    axes[0].set_xlim([start_time, end_time])
    axes[0].set_ylim([0.01, 10])
    axes[0].set_yscale('log')
    axes[0].set_xlabel("Time", fontsize=14)
    axes[0].set_ylabel("Significant Wave Height [m]", fontsize=14)
    axes[0].legend(title="Buoy")
    axes[0].grid(True)

    # Subplot 2: Time series plot for model data (MODHs0)
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        axes[1].scatter(time_series, OMBwave.MODHs0[modelno, buoyno, :], label=f'MET-{buoyno}', color=color, marker='.')

    axes[1].set_xlim([start_time, end_time])
    axes[1].set_ylim([0.01, 10])
    axes[1].set_yscale('log')
    axes[1].set_xlabel("Time", fontsize=14)
    axes[1].set_ylabel("Significant Wave Height [m]", fontsize=14)
    axes[1].legend(title="Model Buoy")
    axes[1].grid(True)

    # Subplot 3: Spectra plot at a specific time
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        specific_time_np = np.datetime64(specific_time)
        time_diffs = np.abs(time_series - specific_time_np)
        tt = np.argmin(time_diffs)

        if time_diffs[tt] <= np.timedelta64(2, 'h'):
            spectra = OMBraw.processed_elevation_energy_spectrum[buoyno, tt, :]
            freqs = OMBraw.frequencies_waves_imu.values
            axes[2].plot(freqs, spectra, label=f"Buoy {buoyno}", color=color)
        else:
            axes[2].plot([], [], label=f"Buoy {buoyno} (No valid time)", color=color)

    axes[2].set_xlabel("Frequency [Hz]", fontsize=14)
    axes[2].set_ylabel("Spectral Energy", fontsize=14)
    axes[2].set_yscale('log')
    axes[2].grid(True)
    axes[2].legend(title="Buoy", loc=4)

    # Subplot 4: Buoy positions at the specific time (with Cartopy map)
    ax = fig.add_subplot(4, 1, 4, projection=ccrs.PlateCarree())  # Using Cartopy projection for this axis
    # Set up a Cartopy map for the region north and west of Svalbard
    ax.set_extent([0, 25, 77, 83], crs=ccrs.PlateCarree())  # Extent covering the area around Svalbard
    ax.coastlines(resolution='10m', color='black', linewidth=1)

    # Plot buoy positions at the specific time
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time[buoyno, :].values
        specific_time_np = np.datetime64(specific_time)
        time_diffs = np.abs(time_series - specific_time_np)
        tt = np.argmin(time_diffs)

        # Only plot position if the time difference is valid
        if time_diffs[tt] <= np.timedelta64(2, 'h'):
            lat = OMBraw.lat[buoyno, tt].values
            lon = OMBraw.lon[buoyno, tt].values
            ax.scatter(lon, lat, label=f"Buoy {buoyno}", color=color, s=100)

    ax.set_xlabel("Longitude", fontsize=14)
    ax.set_ylabel("Latitude", fontsize=14)
    ax.set_title(f"Buoy Positions at {specific_time}")
    ax.grid(True)
    ax.legend(title="Buoy Positions")

    # Adjust aspect ratio for the map
    ax.set_aspect('auto')  # Adjust the aspect ratio to avoid stretching

    plt.tight_layout()
    plt.show()

# Create an interactive widget
time_slider = widgets.SelectionSlider(
    options=pd.date_range(start=start_time, end=end_time, freq='H').strftime('%Y-%m-%d %H:%M').tolist(),
    description='Time:',
    continuous_update=False
)

buoy_list_dropdown = widgets.Dropdown(
    options=['List 1', 'List 2', 'List 3'],
    description='Buoy List:',
    value='List 1'  # Default to the first list
)

# Display the interactive widget
interact(update_plot_with_positions, specific_time_str=time_slider, selected_buoy_list=buoy_list_dropdown)


Widget Javascript not detected.  It may not be installed or enabled properly. Reconnecting the current kernel may help.


<function __main__.update_plot_with_positions(specific_time_str, selected_buoy_list)>

In [3]:
OMBwave.model.values

array(['MF-WAM', 'MET-ARCWAM', 'MET-WAM', 'NRL-COAMPS3', 'NERSC-neXtSIM'],
      dtype=object)

In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from ipywidgets import interact, widgets
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Load datasets
OMBraw = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/cruise/2024_KVS_deployment.nc')
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS23.nc')

# Define the time range for the plot
start_time = pd.to_datetime("2024-04-18")
end_time = pd.to_datetime("2024-04-25")

# Define buoy lists
buoy_list_1 = [30, 7, 1, 17, 32, 10, 8, 23, 18, 26, 9, 13]
buoy_list_2 = [30, 7, 1, 17, 22, 2, 27, 25, 4]
buoy_list_3 = [30, 7, 17, 32, 27, 18, 26]

# Available models in OMBwave
model_options = np.array(OMBwave.model.values).tolist()

# Create a colormap (light blue to dark blue)
colors = cm.Spectral(np.linspace(0, 1, len(buoy_list_1)))  # Default list (buoy_list_1)

# Interactive function to update the plots
def update_plot_with_positions(specific_time_str, selected_buoy_list, selected_model):
    specific_time = pd.to_datetime(specific_time_str)

    # Select buoy list based on the dropdown selection
    if selected_buoy_list == 'List 1':
        buoy_list = buoy_list_1
    elif selected_buoy_list == 'List 2':
        buoy_list = buoy_list_2
    elif selected_buoy_list == 'List 3':
        buoy_list = buoy_list_3

    # Update colors based on the selected list
    colors = cm.Spectral(np.linspace(0, 1, len(buoy_list)))

    # Create a figure with four subplots (adding one for buoy positions)
    fig, axes = plt.subplots(4, 1, figsize=(10, 16), gridspec_kw={'height_ratios': [2, 2, 1, 2]})
    axes = axes.ravel()

    # Subplot 1: Time series plot for raw data (pHs0)
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        axes[0].scatter(time_series, OMBraw.pHs0[buoyno, :], label=str(buoyno), color=color)
    
    stripe_width = pd.Timedelta(hours=1)
    stripe_start = specific_time - stripe_width / 2
    stripe_end = specific_time + stripe_width / 2
    axes[0].add_patch(Rectangle(
        (stripe_start, 0.01),
        stripe_width,
        10,
        color='grey',
        alpha=0.2,
        zorder=0
    ))

    axes[0].set_xlim([start_time, end_time])
    axes[0].set_ylim([0.01, 10])
    axes[0].set_yscale('log')
    axes[0].set_xlabel("Time", fontsize=14)
    axes[0].set_ylabel("Significant Wave Height [m]", fontsize=14)
    axes[0].legend(title="Buoy")
    axes[0].grid(True)

    # Subplot 2: Time series plot for model data (MODHs0)
    model_index = model_options.index(selected_model)  # Find the model index
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        axes[1].scatter(time_series, OMBwave.MODHs0[model_index, buoyno, :], label=f'{selected_model}-{buoyno}', color=color, marker='.')

    axes[1].set_xlim([start_time, end_time])
    axes[1].set_ylim([0.0001, 10])
    axes[1].set_yscale('log')
    axes[1].set_xlabel("Time", fontsize=14)
    axes[1].set_ylabel("Significant Wave Height [m]", fontsize=14)
    #axes[1].legend(title="Model Buoy")
    axes[1].grid(True)

    # Subplot 3: Spectra plot at a specific time
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time_waves_imu[buoyno, :].values
        specific_time_np = np.datetime64(specific_time)
        time_diffs = np.abs(time_series - specific_time_np)
        tt = np.argmin(time_diffs)

        if time_diffs[tt] <= np.timedelta64(2, 'h'):
            spectra = OMBraw.processed_elevation_energy_spectrum[buoyno, tt, :]
            freqs = OMBraw.frequencies_waves_imu.values
            axes[2].plot(freqs, spectra, label=f"Buoy {buoyno}", color=color)
        else:
            axes[2].plot([], [], label=f"Buoy {buoyno} (No valid time)", color=color)

    axes[2].set_xlabel("Frequency [Hz]", fontsize=14)
    axes[2].set_ylabel("Spectral Energy", fontsize=14)
    axes[2].set_yscale('log')
    axes[2].grid(True)
    #axes[2].legend(title="Buoy", loc=4)

    # Subplot 4: Buoy positions at the specific time (with Cartopy map)
    ax = fig.add_subplot(4, 1, 4, projection=ccrs.PlateCarree())  # Using Cartopy projection for this axis
    # Set up a Cartopy map for the region north and west of Svalbard
    ax.set_extent([0, 25, 77, 83], crs=ccrs.PlateCarree())  # Extent covering the area around Svalbard
    ax.coastlines(resolution='10m', color='black', linewidth=1)

    # Plot buoy positions at the specific time
    for i, buoyno in enumerate(buoy_list):
        color = colors[i]
        time_series = OMBraw.time[buoyno, :].values
        specific_time_np = np.datetime64(specific_time)
        time_diffs = np.abs(time_series - specific_time_np)
        tt = np.argmin(time_diffs)

        # Only plot position if the time difference is valid
        if time_diffs[tt] <= np.timedelta64(2, 'h'):
            lat = OMBraw.lat[buoyno, tt].values
            lon = OMBraw.lon[buoyno, tt].values
            ax.scatter(lon, lat, label=f"Buoy {buoyno}", color=color, s=100)

    ax.set_xlabel("Longitude", fontsize=14)
    ax.set_ylabel("Latitude", fontsize=14)
    ax.set_title(f"Buoy Positions at {specific_time}")
    ax.grid(True)
    #ax.legend(title="Buoy Positions")

    # Adjust aspect ratio for the map
    ax.set_aspect('auto')  # Adjust the aspect ratio to avoid stretching

    plt.tight_layout()
    plt.show()

# Create an interactive widget
time_slider = widgets.SelectionSlider(
    options=pd.date_range(start=start_time, end=end_time, freq='H').strftime('%Y-%m-%d %H:%M').tolist(),
    description='Time:',
    continuous_update=False
)

buoy_list_dropdown = widgets.Dropdown(
    options=['List 1', 'List 2', 'List 3'],
    description='Buoy List:',
    value='List 1'  # Default to the first list
)

model_dropdown = widgets.Dropdown(
    options=model_options,
    description='Model:',
    value=model_options[0]  # Default to the first model
)

# Display the interactive widget
interact(update_plot_with_positions, specific_time_str=time_slider, selected_buoy_list=buoy_list_dropdown, selected_model=model_dropdown)


Widget Javascript not detected.  It may not be installed or enabled properly. Reconnecting the current kernel may help.


<function __main__.update_plot_with_positions(specific_time_str, selected_buoy_list, selected_model)>

In [5]:
OMBwave['MODHs0'][5,17,347].values

array(nan)